In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
BlockA — Hindcast O3 (30–70 hPa polar-cap column) scores vs reference
       + reference climatology & anomaly
=====================================================================

新增内容（相对你原来的 BlockA）：
--------------------------------
* 在 reference BWCN 合并模拟上，计算逐日 (day-of-year) 极盖 30–70 hPa O3 柱的
  气候态 O3_ref_clim(doy)。
* 在每个 hindcast case 中：
    - 拿到参考 30–70 hPa 极盖柱 O3_ref(time)
    - 用 day-of-year 映射到 O3_ref_clim(time)
    - 构造：
        O3_ref_anom(time) = O3_ref(time) - O3_ref_clim(time)
        O3_ref_loss(time) = O3_ref_clim(time) - O3_ref(time)
* 上述三个变量写入到每个 O3_scores_{case}.nc 中。

这样 BlockB 就可以统一用 X = O3_ref_loss，Y = {CRPS, bias, spread} 做散点图。
"""

import os
import re
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Tuple, Any

import numpy as np
import xarray as xr

# ---------------------------------------------------------------------
# Paths & constants
# ---------------------------------------------------------------------

HINDCAST_BASE_DIR = "/home/weiji/restart_exam/hindcast_data"
REF_O3_PATH = "/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.merged.nc"
ROOT_OUT_DIR = "/home/weiji/test_code/plots/hindcast/O3_withclim"

os.makedirs(ROOT_OUT_DIR, exist_ok=True)

# ---------------------------------------------------------------------
# Experiment family metadata
# ---------------------------------------------------------------------

MONTH_NAME = {
    "01": "Jan",
    "02": "Feb",
    "03": "Mar",
    "04": "Apr",
    "05": "May",
    "06": "Jun",
    "07": "Jul",
    "08": "Aug",
    "09": "Sep",
    "10": "Oct",
    "11": "Nov",
    "12": "Dec",
}

BASE_FAMILIES: Dict[str, Dict[str, str]] = {
    "": {
        "family": "small_perturbation",
        "short_name": "SP",
        "description": (
            "Small-perturbation ensemble (coupled O3): ~30 members where the "
            "initial temperature fields are perturbed with extremely small "
            "random noise of order 1e-12 K."
        ),
    },
    "V2": {
        "family": "large_perturbation",
        "short_name": "LP",
        "description": (
            "Large-perturbation ensemble (V2): ~30 members where the initial "
            "temperature fields are perturbed with much larger random noise "
            "of order 1e-6 K."
        ),
    },
    "V3": {
        "family": "q_perturbation",
        "short_name": "QP",
        "description": (
            "Q-perturbation ensemble (V3): ~30 members where the initial "
            "water vapour fields are perturbed in a mass-conserving way "
            "(regions with enhanced humidity are compensated by regions "
            "with reduced humidity)."
        ),
    },
}

# ---------------------------------------------------------------------
# Dataclass: one hindcast O3 case
# ---------------------------------------------------------------------

@dataclass
class O3HindcastCase:
    """Container for one hindcast case (O3 ensemble only)."""

    case_dir: Path           # /.../hindcast_data/YYYY-MM*
    case_name: str           # e.g. "0008-02", "0008-02V2", "0008-02-nocouple"
    year_str: str            # "0008"
    month_str: str           # "02"
    tag: str                 # "", "V2", "V3", "-nocouple", ...
    family: str              # small_perturbation / large_perturbation / q_perturbation / initial_prescribed_o3 / unknown
    family_short: str        # SP / LP / QP / NC / UNK
    description: str         # human-readable description
    group_label: str         # e.g. "Feb_couple", "Feb_initial_prescribeO3"
    n_member: int            # number of ensemble members
    member_ids: np.ndarray   # e.g. [1, 2, ..., 30]
    o3: xr.DataArray         # dims: (member, time, plev, lat, lon)

# ---------------------------------------------------------------------
# Parsing & classification of case names (incl. nocouple)
# ---------------------------------------------------------------------

def parse_case_name(case_name: str) -> Dict[str, str]:
    """
    Parse directory name like "0008-02", "0008-02V2", "0008-02V3", "0008-02-nocouple".

    Returns:
        {
            "year_str", "month_str", "tag",
            "family", "family_short", "description", "group_label"
        }
    """
    m = re.match(r"^(?P<year>\d{4})-(?P<month>\d{2})(?P<suffix>.*)$", case_name)
    if not m:
        raise ValueError(f"Case name does not match 'YYYY-MM*' pattern: {case_name}")

    year_str = m.group("year")
    month_str = m.group("month")
    raw_suffix = m.group("suffix")  # may be "", "V2", "V3", "-nocouple", "nocouple", ...

    tag = raw_suffix

    # Default: unknown family
    family = "unknown"
    family_short = "UNK"
    description = (
        f"Unknown perturbation family (suffix='{raw_suffix}') for case '{case_name}'. "
        "Treat as a generic ensemble."
    )

    # Exact matches: "", "V2", "V3"
    if raw_suffix in BASE_FAMILIES:
        meta = BASE_FAMILIES[raw_suffix]
        family = meta["family"]
        family_short = meta["short_name"]
        description = meta["description"]

    # Nocouple: any suffix containing "nocouple"
    elif "nocouple" in raw_suffix.lower():
        family = "initial_prescribed_o3"
        family_short = "NC"
        description = (
            "Nocouple ensemble: same small temperature perturbations as the "
            "small-pert ensemble, but ozone is prescribed (no ozone coupling). "
            "This corresponds to XX_initial_prescribeO3."
        )

    month_label = MONTH_NAME.get(month_str, month_str)

    if family == "small_perturbation" and "nocouple" not in raw_suffix.lower():
        group_label = f"{month_label}_couple"
    elif family == "initial_prescribed_o3":
        group_label = f"{month_label}_initial_prescribeO3"
    elif family == "large_perturbation":
        group_label = f"{month_label}_largePert"
    elif family == "q_perturbation":
        group_label = f"{month_label}_Qpert"
    else:
        group_label = case_name  # fallback if unknown

    return dict(
        year_str=year_str,
        month_str=month_str,
        tag=tag,
        family=family,
        family_short=family_short,
        description=description,
        group_label=group_label,
    )

# ---------------------------------------------------------------------
# Hindcast O3 loaders
# ---------------------------------------------------------------------

def list_case_dirs(base_dir: str | Path) -> List[Path]:
    """List all hindcast case directories under base_dir whose name matches 'YYYY-MM*'."""
    base_dir = Path(base_dir).expanduser().resolve()
    case_dirs: List[Path] = []
    for p in sorted(base_dir.iterdir()):
        if not p.is_dir():
            continue
        if re.match(r"^\d{4}-\d{2}.*$", p.name):
            case_dirs.append(p)
    return case_dirs


def list_o3_files(case_dir: Path) -> List[Path]:
    """List all O3 NetCDF files for a given case (case_dir / 'O3' / '*.nc')."""
    o3_dir = case_dir / "O3"
    if not o3_dir.is_dir():
        return []
    return sorted(o3_dir.glob("*.nc"))


def parse_member_ids_from_filenames(files: List[Path], case_name: str) -> np.ndarray:
    """
    Parse member IDs from filenames.

    Example filename:
        BWCN.e122.f19_g16.007.0008-02.001.small-pertlim.cam.h3.O3.nc
        parts = ["BWCN","e122","f19_g16","007","0008-02","001","small-pertlim",...]

    Strategy:
      - Find the token equal to 'case_name' in the split list.
      - The next token, if purely digits, is treated as the member index.
      - If parsing fails, fall back to 1..N.
    """
    member_ids: List[int] = []

    for idx_file, f in enumerate(files):
        parts = f.name.split(".")
        mem_id: int | None = None

        if case_name in parts:
            i = parts.index(case_name)
            if i + 1 < len(parts) and parts[i + 1].isdigit():
                try:
                    mem_id = int(parts[i + 1])
                except Exception:
                    mem_id = None

        if mem_id is None:
            mem_id = idx_file + 1  # fallback: 1..N

        member_ids.append(mem_id)

    return np.asarray(member_ids, dtype=int)


def load_o3_for_case(case_dir: Path) -> O3HindcastCase | None:
    """
    为单个 hindcast case 读取所有 O3 成员，并统一垂直维到 'plev'。

    返回一个 O3HindcastCase，o3 的维度应为：
        (member, time, plev, lat, lon)
    """
    case_name = case_dir.name
    meta = parse_case_name(case_name)

    files = list_o3_files(case_dir)
    if not files:
        print(f"[INFO] No O3 files found in case {case_name}, skipping.")
        return None

    files = sorted(files)
    member_ids = parse_member_ids_from_filenames(files, case_name)

    print(f"[DEBUG] (new loader) Opening O3 for case {case_name}, members={len(files)}")

    da_list: List[xr.DataArray] = []

    for f, mid in zip(files, member_ids):
        ds_single = xr.open_dataset(f, decode_times=True)
        if "O3" not in ds_single.data_vars:
            print(f"[WARN] File {f} has no O3, skip this member.")
            ds_single.close()
            continue

        da = ds_single["O3"]
        dims = da.dims

        # 统一垂直维名字
        if ("plev_2" in dims) and ("plev" not in dims):
            da = da.rename({"plev_2": "plev"})
        elif ("plev" in dims) and ("plev_2" in dims):
            print(f"[WARN] File {f} has both 'plev' and 'plev_2'; "
                  f"temporarily dropping 'plev_2' by taking index 0.")
            da = da.isel(plev_2=0, drop=True)

        ds_single.close()

        # 增加 member 维
        da = da.expand_dims("member")
        da = da.assign_coords(member=[mid])
        da_list.append(da)

    if len(da_list) == 0:
        print(f"[WARN] No valid O3 DataArray for case {case_name}, skip.")
        return None

    da_o3 = xr.concat(da_list, dim="member")

    print(f"[DEBUG] Combined O3 dims for {case_name}: "
          f"{da_o3.dims}, sizes={dict(da_o3.sizes)}")

    case = O3HindcastCase(
        case_dir=case_dir,
        case_name=case_name,
        year_str=meta["year_str"],
        month_str=meta["month_str"],
        tag=meta["tag"],
        family=meta["family"],
        family_short=meta["family_short"],
        description=meta["description"],
        group_label=meta["group_label"],
        n_member=da_o3.sizes["member"],
        member_ids=da_o3["member"].values,
        o3=da_o3,
    )
    return case


def load_all_o3_hindcasts(base_dir: str | Path) -> Tuple[List[O3HindcastCase], Dict[str, List[O3HindcastCase]]]:
    base_dir = Path(base_dir).expanduser().resolve()
    all_cases: List[O3HindcastCase] = []
    by_family: Dict[str, List[O3HindcastCase]] = {}

    case_dirs = list_case_dirs(base_dir)
    print(f"[INFO] Found {len(case_dirs)} candidate case directories under {base_dir}")

    for case_dir in case_dirs:
        case = load_o3_for_case(case_dir)
        if case is None:
            continue

        all_cases.append(case)
        fam = case.family
        by_family.setdefault(fam, []).append(case)

        print(
            f"[LOADED] {case.case_name:18s}  "
            f"family={case.family_short:3s}  "
            f"group={case.group_label:25s}  "
            f"members={case.n_member:2d}"
        )

    print("[INFO] Finished loading O3 hindcasts.")
    for fam, lst in by_family.items():
        print(f"  - {fam:22s}: {len(lst)} cases")

    return all_cases, by_family

# ---------------------------------------------------------------------
# 30–70 hPa partial column O3 (DU) & polar-cap mean
# ---------------------------------------------------------------------

def calc_parc_o3_du(var, p_top=30, p_bottom=70):
    """
    计算 p_top–p_bottom hPa 的部分臭氧柱（DU）。
    支持带 member 维；垂直坐标可为 plev / lev / level / plev_2。
    """
    m_air = 28.964 / (6.022E23)
    g = 980.6

    # 垂直维识别
    if 'plev' in var.dims:
        plev = var.plev; level = 'plev'
    elif 'plev_2' in var.dims:
        plev = var.plev_2; level = 'plev_2'
    elif 'lev' in var.dims:
        plev = var.lev; level = 'lev'
    elif 'level' in var.dims:
        plev = var.level; level = 'level'
    else:
        raise ValueError('No pressure level coordinate found in data.')

    time = var.time
    delta_p = np.zeros((len(time), len(plev)))

    # 单位判断
    if plev[0] < plev[-1] and plev[-1] <= 1000:
        factor, factor_2 = 100, 1
    elif plev[0] > plev[-1] and plev[0] <= 1000:
        factor, factor_2 = 100, 1
    elif plev[0] < plev[-1] and plev[-1] > 1000:
        factor, factor_2 = 1, 100
    else:
        factor, factor_2 = 1, 100

    # 升序
    if plev[0] < plev[-1]:
        for i in range(1, len(plev)):
            delta_p[:, i] = plev[i] - plev[i-1]

        weights_p = xr.DataArray(
            delta_p * factor,
            dims=['time', level],
            coords=[time, plev]
        )
        O3 = var * weights_p * 10 / (g * m_air)

        O3 = O3.sel(**{level: slice(p_top * factor_2, p_bottom * factor_2)})
        O3 = O3.sum(dim=level) / 2.687E16

    # 降序
    else:
        for i in range(len(plev) - 1):
            delta_p[:, i] = plev[i] - plev[i+1]

        weights_p = xr.DataArray(
            delta_p * factor,
            dims=['time', level],
            coords=[time, plev]
        )
        O3 = var * weights_p * 10 / (g * m_air)

        O3 = O3.sel(**{level: slice(p_bottom * factor_2, p_top * factor_2)})
        O3 = O3.sum(dim=level) / 2.687E16

    return O3.where(O3 != 0)


def polar_cap_mean_30_70(o3_4d: xr.DataArray,
                         lat_min: float = 60.0,
                         lat_max: float = 90.0) -> xr.DataArray:
    """
    给定 O3 四维场（可以包含 member 维），计算：
      1) 沿 lon 做经向平均
      2) 30–70 hPa 部分臭氧柱（DU）
      3) 60–90N 极盖余弦加权平均

    返回：
      - 若有 member 维：dims = (member, time)
      - 若无 member 维：dims = (time,)
    """
    if 'lon' in o3_4d.dims:
        o3_zm = o3_4d.mean(dim='lon')
    else:
        o3_zm = o3_4d

    if 'plev_2' in o3_zm.dims and 'plev' not in o3_zm.dims:
        o3_zm = o3_zm.rename({'plev_2': 'plev'})

    o3_pc = calc_parc_o3_du(o3_zm, p_top=30, p_bottom=70)
    o3_cap = o3_pc.sel(lat=slice(lat_min, lat_max))
    weights = np.cos(np.deg2rad(o3_cap.lat))
    o3_cap = o3_cap.weighted(weights).mean(dim='lat')

    return o3_cap

# ---------------------------------------------------------------------
# CRPS / spread / mean error 子程序
# ---------------------------------------------------------------------

def crps_ensemble(ensemble: np.ndarray, obs: np.ndarray) -> np.ndarray:
    ens = np.asarray(ensemble)
    obs_arr = np.asarray(obs)

    if ens.ndim < 1:
        raise ValueError("ensemble must have at least 1 dimension: (member, ...)")

    diff_to_obs = np.abs(ens - obs_arr)
    term1 = diff_to_obs.mean(axis=0)

    diff_ij = np.abs(ens[:, None, ...] - ens[None, :, ...])
    term2 = 0.5 * diff_ij.mean(axis=(0, 1))

    return term1 - term2


def ensemble_spread(ensemble: np.ndarray) -> np.ndarray:
    ens = np.asarray(ensemble)
    return ens.std(axis=0, ddof=0)


def ensemble_bias(ensemble: np.ndarray, obs: np.ndarray) -> np.ndarray:
    ens = np.asarray(ensemble)
    obs_arr = np.asarray(obs)
    return ens.mean(axis=0) - obs_arr


def ensemble_mae(ensemble: np.ndarray, obs: np.ndarray) -> np.ndarray:
    ens = np.asarray(ensemble)
    obs_arr = np.asarray(obs)
    return np.abs(ens - obs_arr).mean(axis=0)

# ---------------------------------------------------------------------
# Reference O3 loader & climatology
# ---------------------------------------------------------------------

def load_ref_o3_polarcap(ref_path: str | Path = REF_O3_PATH) -> xr.DataArray:
    """
    从 BWCN merged 文件中读取 O3，并计算 30–70 hPa 极盖平均（60–90N）臭氧柱（DU）。

    返回：
        DataArray: dims = ('time',)，单位 DU。
    """
    ref_path = Path(ref_path).expanduser().resolve()
    print(f"[REF] Opening reference file: {ref_path}")

    ds_ref = xr.open_dataset(ref_path, decode_times=True)
    print(f"[REF] Reference dims: {dict(ds_ref.dims)}")

    if "O3" not in ds_ref.data_vars:
        raise KeyError(f"O3 not found in reference dataset. Vars: {list(ds_ref.data_vars)}")

    o3_ref = ds_ref["O3"]
    print(f"[REF] O3 dims: {o3_ref.dims}, sizes={dict(o3_ref.sizes)}")
    print(f"[REF] time range: {o3_ref.time.min().values} -> {o3_ref.time.max().values}")

    col_ref = polar_cap_mean_30_70(o3_ref)  # dims: (time,)
    col_ref.name = "O3_pc_60_90N_30_70hPa"

    return col_ref


def compute_daily_climatology(ref_col: xr.DataArray) -> xr.DataArray:
    """
    在 BWCN merged 的参考时间序列上计算逐日气候态：
        clim(doy) = mean_{years} [O3_ref(time) | dayofyear = doy]
    """
    clim = ref_col.groupby("time.dayofyear").mean("time")
    clim.name = "O3_ref_clim_daily"
    return clim

# ---------------------------------------------------------------------
# 计算单个 hindcast case 的各项指标
# ---------------------------------------------------------------------

def compute_scores_for_case(case: O3HindcastCase,
                            ref_col_all: xr.DataArray,
                            ref_clim_doy: xr.DataArray) -> xr.Dataset:
    """
    对单个 hindcast case 计算：
        - O3_cap(member, time)
        - O3_ref(time)      : 参考极盖柱（DU）
        - O3_ref_clim(time) : 逐日气候态（由 day-of-year 映射）
        - O3_ref_anom(time) : O3_ref - clim
        - O3_ref_loss(time) : clim - O3_ref
        - CRPS(time)
        - spread(time)
        - bias(time)
        - mae(time)
    """
    print(f"\n[CASE] Processing case: {case.case_name} ({case.group_label})")
    print(f"       family={case.family}  members={case.n_member}")

    # 1. hindcast 极盖柱
    o3_cap = polar_cap_mean_30_70(case.o3)  # dims: (member, time)
    print(f"[DEBUG] O3_cap dims: {o3_cap.dims}, sizes={dict(o3_cap.sizes)}")
    print(f"[DEBUG] O3_cap time range: {o3_cap.time.min().values} -> {o3_cap.time.max().values}")

    # 2. 与参考对齐时间轴
    o3_cap_aligned, ref_aligned = xr.align(o3_cap, ref_col_all, join="inner")
    print(f"[DEBUG] After align: hindcast time len={o3_cap_aligned.time.size}, "
          f"ref time len={ref_aligned.time.size}")

    if o3_cap_aligned.time.size == 0:
        raise RuntimeError(f"No overlapping time between hindcast {case.case_name} and reference.")

    # 3. 参考气候态 & anomaly / loss
    #    dayofyear 1..365（无闰年）
    doy = o3_cap_aligned.time.dt.dayofyear
    ref_clim_oncase = ref_clim_doy.sel(dayofyear=doy)
    ref_clim_oncase = ref_clim_oncase.rename("O3_ref_clim")
    ref_clim_oncase = ref_clim_oncase.assign_coords(time=o3_cap_aligned.time)

    O3_ref = ref_aligned.rename("O3_ref")
    O3_ref_anom = (O3_ref - ref_clim_oncase).rename("O3_ref_anom")
    O3_ref_loss = (ref_clim_oncase - O3_ref).rename("O3_ref_loss")

    # 4. 转 numpy 计算评分
    ens_np = o3_cap_aligned.values  # (member, time)
    obs_np = O3_ref.values          # (time,)

    crps = crps_ensemble(ens_np, obs_np)
    spread = ensemble_spread(ens_np)
    bias = ensemble_bias(ens_np, obs_np)
    mae = ensemble_mae(ens_np, obs_np)

    # 5. lead（0-based） + 其他坐标
    n_time = o3_cap_aligned.time.size
    lead = np.arange(n_time, dtype=int)

    ds_out = xr.Dataset(
        coords={
            "time": o3_cap_aligned.time,
            "lead": ("time", lead),
            "member": o3_cap_aligned.member,
        },
        data_vars={
            "O3_cap": (("member", "time"), ens_np),
            "O3_ref": (("time",), O3_ref.values),
            "O3_ref_clim": (("time",), ref_clim_oncase.values),
            "O3_ref_anom": (("time",), O3_ref_anom.values),
            "O3_ref_loss": (("time",), O3_ref_loss.values),
            "CRPS": (("time",), crps),
            "spread": (("time",), spread),
            "bias": (("time",), bias),
            "mae": (("time",), mae),
        },
        attrs={
            "case_name": case.case_name,
            "year_str": case.year_str,
            "month_str": case.month_str,
            "group_label": case.group_label,
            "family": case.family,
            "family_short": case.family_short,
            "description": case.description,
            "note": (
                "30–70 hPa polar-cap (60–90N) O3 column in DU; "
                "scores (CRPS, spread, bias, mae) computed daily "
                "from hindcast start, relative to reference BWCN "
                "merged simulation; O3_ref_clim / anom / loss derived "
                "from BWCN daily climatology."
            ),
        },
    )

    # 单位
    for v in ["O3_cap", "O3_ref", "O3_ref_clim", "O3_ref_anom", "O3_ref_loss",
              "CRPS", "spread", "bias", "mae"]:
        ds_out[v].attrs["units"] = "DU"

    return ds_out

# ---------------------------------------------------------------------
# BlockA 主函数：只处理 year=0008 的所有 case
# ---------------------------------------------------------------------

def main_blockA():
    print("[BlockA] Starting hindcast O3 BlockA ...")
    print(f"[BlockA] Hindcast base dir: {HINDCAST_BASE_DIR}")
    print(f"[BlockA] Reference file   : {REF_O3_PATH}")
    print(f"[BlockA] Output root      : {ROOT_OUT_DIR}")

    # 1. 所有 hindcast case
    cases, by_family = load_all_o3_hindcasts(HINDCAST_BASE_DIR)

    # 2. 只保留 year=0008
    cases_0008 = [c for c in cases if c.year_str == "0008"]
    print(f"\n[BlockA] Found {len(cases_0008)} hindcast cases for year 0008.")
    for c in cases_0008:
        print(f"         - {c.case_name:18s}  group={c.group_label:25s}  family={c.family_short}")

    if not cases_0008:
        print("[BlockA] No year 0008 cases found. Nothing to do.")
        return

    # 3. 参考极盖柱 & 气候态
    ref_col_all = load_ref_o3_polarcap(REF_O3_PATH)  # 全年多年的 TS
    print(f"[BlockA] Reference polar-cap O3 time len={ref_col_all.time.size}")
    print(f"[BlockA] Reference time range: {ref_col_all.time.min().values} -> {ref_col_all.time.max().values}")

    ref_clim_doy = compute_daily_climatology(ref_col_all)
    print(f"[BlockA] Computed daily climatology with {ref_clim_doy.sizes['dayofyear']} days")

    # 4. 对每一个 year-0008 case 计算 scores
    for case in cases_0008:
        ds_scores = compute_scores_for_case(case, ref_col_all, ref_clim_doy)

        out_path = os.path.join(ROOT_OUT_DIR, f"O3_scores_{case.case_name}.nc")
        ds_scores.to_netcdf(out_path)
        print(f"[BlockA] Saved scores for {case.case_name} to: {out_path}")

    print("\n[BlockA] All year-0008 hindcast cases processed. Done.")


if __name__ == "__main__":
    main_blockA()


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

# ============================================================
# BlockB — O3 散点图：metric vs O3_ref_loss  （CRPS / bias / spread）
#
# 说明：
#   * 读取 BlockA 生成的 O3_scores_{case_name}.nc
#   * 对每个 group 画 3 张图：
#       X 轴:  O3_ref_loss = O3_ref_clim - O3_ref (DU)
#       Y 轴:  metric ∈ {CRPS, bias, spread}
#   * 每个 case：
#       - 散点
#       - 线性拟合直线 (y = a x + b)
#       - 图例中注明该 case 的相关系数 r
#   * 数据筛选：
#       1) lead ∈ [LEAD_START_DEFAULT, LEAD_END_DEFAULT] （天）
#       2) month ∈ [1..5] (Jan–May)
#       3) month ∈ group 内所有 case 公共月份交集
#       4) time ≥ init_time + 30 天（按各自 time 轴来算）
#
# 输出目录：
#   /home/weiji/test_code/plots/hindcast/O3_scatter
# ============================================================

import os
from typing import Dict, Any, List

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

# ----------------- 基本路径与参数 -----------------
SCORES_ROOT_DIR = "/home/weiji/test_code/plots/hindcast/O3_withclim"
FIG_OUT_DIR     = "/home/weiji/test_code/plots/hindcast/O3_scatter"
os.makedirs(FIG_OUT_DIR, exist_ok=True)

LEAD_START_DEFAULT = 1
LEAD_END_DEFAULT   = 150   # 使用 1–150 天

# ----------------- 要绘制的指标配置 -----------------
METRICS: List[Dict[str, Any]] = [
    {
        "var_name": "CRPS",
        "pretty":   "CRPS",
        "ylabel":   "CRPS (polar-cap 30–70 hPa O$_3$) [DU]",
        "file_prefix": "CRPS",
    },
    {
        "var_name": "bias",
        "pretty":   "Mean error",
        "ylabel":   "Mean error (ensemble mean $-$ reference) [DU]",
        "file_prefix": "bias",
    },
    {
        "var_name": "spread",
        "pretty":   "Ensemble spread",
        "ylabel":   "Ensemble spread (std across members) [DU]",
        "file_prefix": "spread",
    },
]

# ----------------- 分组配置（与你之前一样） -----------------
GROUPS: List[Dict[str, Any]] = [
    # 组0：small-pert，不同 initial month
    {
        "group_id": 0,
        "name": "group0_small_different_init_month",
        "title": "Group 0 – small perturbation, different initialization month",
        "cases": [
            {"case_name": "0008-01", "label": "Jan small (chem) 0008-01", "color": "C0"},
            {"case_name": "0008-02", "label": "Feb small (chem) 0008-02", "color": "C1"},
            {"case_name": "0008-03", "label": "Mar small (chem) 0008-03", "color": "C2"},
        ],
    },
    # 组1：Feb chem vs nocouple
    {
        "group_id": 1,
        "name": "group1_Feb_chem_vs_nochem",
        "title": "Group 1 – Feb initialization: chem vs prescribe-O3",
        "cases": [
            {"case_name": "0008-02",         "label": "Feb chem (0008-02)",         "color": "C0"},
            {"case_name": "0008-02_NOCOUPL", "label": "Feb prescribe-O3 (NOCOUPL)", "color": "C3"},
        ],
    },
    # 组2：Mar chem vs nocouple
    {
        "group_id": 2,
        "name": "group2_Mar_chem_vs_nochem",
        "title": "Group 2 – Mar initialization: chem vs prescribe-O3",
        "cases": [
            {"case_name": "0008-03",         "label": "Mar chem (0008-03)",         "color": "C0"},
            {"case_name": "0008-03_NOCOUPL", "label": "Mar prescribe-O3 (NOCOUPL)", "color": "C3"},
        ],
    },
    # 组3：large-pert，不同 initial month
    {
        "group_id": 3,
        "name": "group3_largepert_different_init_month",
        "title": "Group 3 – large perturbation, different initialization month",
        "cases": [
            {"case_name": "0008-02_v2", "label": "Feb large-pert (0008-02_v2)", "color": "C0"},
            {"case_name": "0008-03_v2", "label": "Mar large-pert (0008-03_v2)", "color": "C1"},
            {"case_name": "0008-04_v2", "label": "Apr large-pert (0008-04_v2)", "color": "C2"},
        ],
    },
    # 组4：Mar init，不同 pert
    {
        "group_id": 4,
        "name": "group4_Mar_different_pert",
        "title": "Group 4 – Mar initialization, different perturbations",
        "cases": [
            {"case_name": "0008-03",    "label": "Mar small-pert (0008-03)",    "color": "C0"},
            {"case_name": "0008-03_v2", "label": "Mar large-pert (0008-03_v2)", "color": "C1"},
            {"case_name": "0008-03_v3", "label": "Mar Q-pert (0008-03_v3)",     "color": "C2"},
        ],
    },
    # 组5：Q-pert，不同 init month
    {
        "group_id": 5,
        "name": "group5_Qpert_different_init_month",
        "title": "Group 5 – Q-pert, different initialization month",
        "cases": [
            {"case_name": "0008-02_v3", "label": "Feb Q-pert (0008-02_v3)", "color": "C0"},
            {"case_name": "0008-03_v3", "label": "Mar Q-pert (0008-03_v3)", "color": "C1"},
        ],
    },
    # 组6：Feb init，不同 pert
    {
        "group_id": 6,
        "name": "group6_Feb_different_pert",
        "title": "Group 6 – Feb initialization, different perturbations",
        "cases": [
            {"case_name": "0008-02",    "label": "Feb small-pert (0008-02)",    "color": "C0"},
            {"case_name": "0008-02_v2", "label": "Feb large-pert (0008-02_v2)", "color": "C1"},
            {"case_name": "0008-02_v3", "label": "Feb Q-pert (0008-02_v3)",     "color": "C2"},
        ],
    },
]

# ============================================================
# 一、从 scores 文件加载散点所需数据
# ============================================================

def _load_case_for_scatter(
    case_name: str,
    metric_name: str,
    root_dir: str = SCORES_ROOT_DIR,
):
    """
    从 O3_scores_{case_name}.nc 中读取：
        metric(time)       : metric_name ∈ {CRPS, bias, spread}
        O3_ref_loss(time)  : clim - ref
        lead(time)         : 0,1,2,...
        time(time)
    """
    nc_path = os.path.join(root_dir, f"O3_scores_{case_name}.nc")
    if not os.path.exists(nc_path):
        print(f"[Scatter] WARNING: file not found for {case_name}: {nc_path}")
        return None

    ds = xr.open_dataset(nc_path)

    needed = [metric_name, "O3_ref_loss"]
    for v in needed:
        if v not in ds.data_vars:
            print(
                f"[Scatter] WARNING: '{v}' not in {nc_path}. "
                f"Available vars: {list(ds.data_vars)}"
            )
            ds.close()
            return None

    metric = ds[metric_name].sortby("time")
    loss   = ds["O3_ref_loss"].sortby("time")

    if "lead" in ds.coords:
        lead = ds["lead"].sortby("time")
    else:
        n_time = metric.sizes["time"]
        lead = xr.DataArray(
            np.arange(n_time, dtype=int),
            coords={"time": metric.time},
            dims=["time"],
            name="lead",
        )

    time = metric.time
    ds.close()

    return {
        "metric": metric,
        "loss": loss,
        "lead": lead,
        "time": time,
    }

# 一个小工具：对 cftime / numpy.datetime64 都能加 30 天
def _add_30_days(init_time):
    """
    尝试对 init_time 加 30 天，兼容 numpy.datetime64, cftime, python datetime.
    """
    # numpy.datetime64
    if isinstance(init_time, np.datetime64):
        return init_time + np.timedelta64(30, "D")

    # cftime or python datetime
    from datetime import timedelta
    try:
        return init_time + timedelta(days=30)
    except Exception:
        # 最后兜底：直接返回原值（后面会比较失败并跳过该 case）
        return init_time

# ============================================================
# 二、按 group 画 metric vs O3_ref_loss 的散点 + 拟合线
# ============================================================

def plot_group_scatter_metric_vs_o3loss(
    group_cfg: Dict[str, Any],
    metric_cfg: Dict[str, Any],
    scores_root: str = SCORES_ROOT_DIR,
    fig_dir: str = FIG_OUT_DIR,
    lead_start: int = LEAD_START_DEFAULT,
    lead_end: int = LEAD_END_DEFAULT,
):
    """
    为一个 group + 一个 metric 画散点：
        X: O3_ref_loss = O3_ref_clim - O3_ref
        Y: metric_name ∈ {CRPS, bias, spread}
    """
    os.makedirs(fig_dir, exist_ok=True)

    gid    = group_cfg["group_id"]
    gname  = group_cfg["name"]
    gtitle = group_cfg["title"]
    cases  = group_cfg["cases"]

    metric_name = metric_cfg["var_name"]
    pretty      = metric_cfg["pretty"]
    ylabel      = metric_cfg["ylabel"]
    file_prefix = metric_cfg["file_prefix"]

    print(f"\n[Scatter] ==== Group {gid} ({gname}), metric={metric_name} ====")

    case_data_list = []
    common_months = set(range(1, 6))  # 先假定 1–5 月

    # -------- 1) 先扫一遍，确定公共月份 & 缓存数据 --------
    for cfg in cases:
        case_name = cfg["case_name"]
        data = _load_case_for_scatter(
            case_name=case_name,
            metric_name=metric_name,
            root_dir=scores_root,
        )
        if data is None:
            print(f"[Scatter]  case {case_name}: load failed, skip.")
            continue

        metric = data["metric"]
        loss   = data["loss"]
        lead   = data["lead"]
        time   = data["time"]

        lead_vals = lead.values
        months    = time.dt.month.values.astype(int)
        tvals     = time.values

        init_time = tvals.min()
        t_threshold = _add_30_days(init_time)

        try:
            mask_after_init = tvals >= t_threshold
        except Exception as e:
            print(f"[Scatter]  case {case_name}: compare t>=t_threshold failed: {e}")
            continue

        mask_lead = (lead_vals >= (lead_start - 1)) & (lead_vals <= (lead_end - 1))
        mask_month_jan_may = (months >= 1) & (months <= 5)

        mask_base = mask_lead & mask_month_jan_may & mask_after_init
        months_this_case = set(np.unique(months[mask_base]))

        print(
            f"[Scatter]  case {case_name}: "
            f"init_time={init_time}, threshold={t_threshold}, "
            f"months_base={sorted(months_this_case)}, n_base={mask_base.sum()}"
        )

        if len(months_this_case) == 0:
            continue

        common_months &= months_this_case
        case_data_list.append(
            {
                "cfg": cfg,
                "metric": metric,
                "loss": loss,
                "lead": lead,
                "time": time,
                "init_time": init_time,
                "t_threshold": t_threshold,
            }
        )

    # 限定在 1–5 月
    common_months = {m for m in common_months if 1 <= m <= 5}

    if not case_data_list:
        print(f"[Scatter]  Group {gid}: no valid cases, skip.")
        return

    if not common_months:
        print(
            f"[Scatter]  Group {gid}: no common months among cases "
            f"(within Jan–May, lead range, and >=1 month after init); skip."
        )
        return

    common_months_sorted = sorted(common_months)
    month_label_map = {1: "Jan", 2: "Feb", 3: "Mar", 4: "Apr", 5: "May"}
    common_months_str = ", ".join(
        month_label_map.get(m, str(m)) for m in common_months_sorted
    )
    print(
        f"[Scatter]  Group {gid} common months: {common_months_sorted} "
        f"({common_months_str})"
    )

    # -------- 2) 真正画散点 + 拟合线 --------
    fig, ax = plt.subplots(figsize=(7.5, 5.5))

    all_x = []
    all_y = []

    for entry in case_data_list:
        cfg        = entry["cfg"]
        case_name  = cfg["case_name"]
        metric     = entry["metric"]
        loss       = entry["loss"]
        lead       = entry["lead"]
        time       = entry["time"]
        t_threshold = entry["t_threshold"]

        lead_vals = lead.values
        months    = time.dt.month.values.astype(int)
        tvals     = time.values

        try:
            mask_after_init = tvals >= t_threshold
        except Exception as e:
            print(f"[Scatter]  case {case_name}: compare in plotting failed: {e}")
            continue

        mask_lead = (lead_vals >= (lead_start - 1)) & (lead_vals <= (lead_end - 1))
        mask_month_jan_may = (months >= 1) & (months <= 5)
        mask_common_month  = np.isin(months, list(common_months))

        mask = mask_lead & mask_month_jan_may & mask_common_month & mask_after_init

        if not np.any(mask):
            print(f"[Scatter]  case {case_name}: no points after full mask.")
            continue

        x_arr = loss.values[mask]
        y_arr = metric.values[mask]

        valid = np.isfinite(x_arr) & np.isfinite(y_arr)
        x_valid = x_arr[valid]
        y_valid = y_arr[valid]

        print(
            f"[Scatter]  case {case_name}: "
            f"n_raw={x_arr.size}, n_valid={x_valid.size}"
        )

        if x_valid.size < 2:
            print(f"[Scatter]  case {case_name}: not enough valid points.")
            continue

        # 相关系数 & 线性拟合
        r_case = np.corrcoef(x_valid, y_valid)[0, 1]
        slope, intercept = np.polyfit(x_valid, y_valid, 1)

        print(
            f"[Scatter]  case {case_name}: "
            f"r={r_case:.3f}, slope={slope:.3f}, intercept={intercept:.3f}"
        )

        # 散点
        ax.scatter(
            x_valid,
            y_valid,
            label=None,
            color=cfg.get("color", None),
            s=18,
            alpha=0.6,
            edgecolors="none",
        )

        # 拟合线
        x_min, x_max = x_valid.min(), x_valid.max()
        x_line = np.linspace(x_min, x_max, 100)
        y_line = slope * x_line + intercept

        ax.plot(
            x_line,
            y_line,
            color=cfg.get("color", None),
            linewidth=2,
            label=f"{cfg['label']} (r={r_case:.2f})",
        )

        all_x.append(x_valid)
        all_y.append(y_valid)

    if not all_x:
        print(f"[Scatter]  Group {gid}: all cases filtered out, skip.")
        plt.close(fig)
        return

    all_x_concat = np.concatenate(all_x)
    all_y_concat = np.concatenate(all_y)
    r_group = np.corrcoef(all_x_concat, all_y_concat)[0, 1]
    print(
        f"[Scatter]  Group {gid}: overall n={all_x_concat.size}, "
        f"r_group={r_group:.3f}"
    )

    # 轴标签 & 标题
    x_label = (
        "Reference ozone loss vs climatology "
        r"$\Delta O_3 = O_{3,\mathrm{clim}} - O_{3,\mathrm{ref}}$ [DU]"
    )
    ax.set_xlabel(x_label, fontsize=12)
    ax.set_ylabel(ylabel, fontsize=12)

    ax.set_title(
        gtitle
        + "\n"
        + f"Scatter: {pretty} vs O$_3$ climatological loss "
        f"(lead {lead_start}–{lead_end} days, months: {common_months_str})\n"
        + "Only points ≥ 30 days after each case's initialization\n"
        + f"Group-wise correlation: r = {r_group:.2f}",
        fontsize=11,
    )

    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=9)
    fig.tight_layout()

    base_name = (
        f"{file_prefix}_scatter_O3loss_group{gid}_{gname}"
        f"_lead{lead_start}_{lead_end}_after30days"
    )
    out_png = os.path.join(fig_dir, base_name + ".png")
    out_pdf = os.path.join(fig_dir, base_name + ".pdf")
    fig.savefig(out_png, dpi=300)
    fig.savefig(out_pdf, dpi=300)

    print(f"[Scatter]  Saved scatter for group {gid}, metric={metric_name} to: {out_png}")

# ============================================================
# 三、主入口：对所有 group × metric 画散点
# ============================================================

def main_blockB():
    print(
        "[Scatter] Start scatter plots: "
        "metrics {CRPS, bias, spread} vs O3_ref_loss for all groups "
        "(>=30 days after init) ..."
    )

    for metric_cfg in METRICS:
        metric_name = metric_cfg["var_name"]
        print(f"\n[Scatter] === Metric: {metric_name} ===")
        for g in GROUPS:
            plot_group_scatter_metric_vs_o3loss(
                group_cfg=g,
                metric_cfg=metric_cfg,
                scores_root=SCORES_ROOT_DIR,
                fig_dir=FIG_OUT_DIR,
                lead_start=LEAD_START_DEFAULT,
                lead_end=LEAD_END_DEFAULT,
            )

    print("[Scatter] All scatter plots done.")


if __name__ == "__main__":
    main_blockB()


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
BlockC — Member-wise correlation: U60N'(t,z) vs seasonal O3' (per experiment & grouped)
=======================================================================================

相关性定义（*按成员*）：
  对固定实验 E、固定 (t,z)，用成员 m=1..M 作为样本：
    X_m(t,z) = U'_{m}(t,z) / sigma_U(z)
    Y_m^MAM  = <O3'_{m}>_{Mar–May} / sigma_O3
    Y_m^AM   = <O3'_{m}>_{Apr–May} / sigma_O3

    r_MAM_E(t,z) = corr_m( X_m(t,z), Y_m^MAM )
    r_AM_E (t,z) = corr_m( X_m(t,z), Y_m^AM  )

对一个 group（包含若干实验）：再把这些实验的成员全部拼在一起算“合并相关”：
    r_MAM_group(t,z) = corr_{(E,m)}( X_{E,m}(t,z), Y_{E,m}^MAM )
    r_AM_group (t,z) = corr_{(E,m)}( X_{E,m}(t,z), Y_{E,m}^AM  )

目标：
  - 处理 group0–group6（与 BlockB 中的分组保持一致）：
      group0_small_different_init_month
      group1_Feb_chem_vs_nochem
      group2_Mar_chem_vs_nochem
      group3_largepert_different_init_month
      group4_Mar_different_pert
      group5_Qpert_different_init_month
      group6_Feb_different_pert
  - 3–5 月 (MAM) & 4–5 月 (AM) 两种季节平均 O3'

输出：
  /home/weiji/test_code/plots/hindcast/UO3_corr/UO3_corr_group{gid}.nc

  结构：
    coords:
      time(time)
      lev(lev)     — Pa
      case(case)   — 每个 group 内的实验名（如 '0008-02', '0008-02_NOCOUPL', ...）

    data_vars:
      corr_MAM_case(case, time, lev)
      corr_AM_case (case, time, lev)
      corr_MAM_group(time, lev)
      corr_AM_group (time, lev)
      n_member_case(case)      — 每个实验的成员数
      n_samples_group          — group 内全部成员总数（用于 BlockD 标题和显著性）

    attrs:
      group_id, group_name, case_names
"""

import os
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import xarray as xr

# ---------------------------------------------------------------------
# 路径配置
# ---------------------------------------------------------------------

# hindcast U 原始数据
HINDCAST_BASE_DIR = "/home/weiji/restart_exam/hindcast_data"

# U 气候态 / 参考
U_ROOT = "/home/weiji/test_code/plots/New_weiji_general_why0008important/General_U"
U_EXTR_60N_NC = os.path.join(U_ROOT, "U_EXTR_60N_lev_time.nc")

# O3 气候态 / 参考
O3_ROOT = "/home/weiji/test_code/plots/New_weiji_general_why0008important/General_O3"
O3_EXTR_PC_NC = os.path.join(O3_ROOT, "O3_pc_EXTR_60_90N_30_70hPa_200yrs.nc")

# O3 hindcast scores（BlockA O3 的输出）
HINDCAST_O3_SCORES_DIR = "/home/weiji/test_code/plots/hindcast/O3"

# 本 Block 输出目录
CORR_ROOT = "/home/weiji/test_code/plots/hindcast/UO3_corr"
os.makedirs(CORR_ROOT, exist_ok=True)


# ---------------------------------------------------------------------
# Group 定义：与 BlockB 的 0–6 组保持一致
# ---------------------------------------------------------------------

GROUPS = [
    {
        "group_id": 0,
        "group_name": "group0_small_different_init_month",
        "cases": [
            "0008-01",  # Jan small (chem)
            "0008-02",  # Feb small (chem)
            "0008-03",  # Mar small (chem)
        ],
    },
    {
        "group_id": 1,
        "group_name": "group1_Feb_chem_vs_nochem",
        "cases": [
            "0008-02",          # Feb chem
            "0008-02_NOCOUPL",  # Feb prescribe-O3 (NOCOUPL)
        ],
    },
    {
        "group_id": 2,
        "group_name": "group2_Mar_chem_vs_nochem",
        "cases": [
            "0008-03",          # Mar chem
            "0008-03_NOCOUPL",  # Mar prescribe-O3 (NOCOUPL)
        ],
    },
    {
        "group_id": 3,
        "group_name": "group3_largepert_different_init_month",
        "cases": [
            "0008-02_v2",  # Feb large-pert
            "0008-03_v2",  # Mar large-pert
            "0008-04_v2",  # Apr large-pert
        ],
    },
    {
        "group_id": 4,
        "group_name": "group4_Mar_different_pert",
        "cases": [
            "0008-03",    # Mar small-pert
            "0008-03_v2", # Mar large-pert
            "0008-03_v3", # Mar Q-pert
        ],
    },
    {
        "group_id": 5,
        "group_name": "group5_Qpert_different_init_month",
        "cases": [
            "0008-02_v3",  # Feb Q-pert
            "0008-03_v3",  # Mar Q-pert
        ],
    },
    {
        "group_id": 6,
        "group_name": "group6_Feb_different_pert",
        "cases": [
            "0008-02",    # Feb small-pert
            "0008-02_v2", # Feb large-pert
            "0008-02_v3", # Feb Q-pert
        ],
    },
]


# ---------------------------------------------------------------------
# 1. U hindcast loader（保留 member 维）
# ---------------------------------------------------------------------

def load_and_process_single_member(fpath: Path, member_id: int) -> xr.DataArray | None:
    """
    读取单个成员，清洗维度，计算 U @ 60N。
    返回：
      DataArray(time, lev)，坐标 lev 为 Pa。
    """
    try:
        ds = xr.open_dataset(fpath, decode_times=True)

        if "U" not in ds:
            print(f"    [WARN] No variable 'U' in {fpath.name}")
            ds.close()
            return None

        da = ds["U"]

        # 处理 plev_2 / ilev
        if "plev_2" in da.dims:
            if "plev" in da.dims:
                da = da.isel(plev_2=0, drop=True)
            else:
                da = da.rename({"plev_2": "plev"})

        if "ilev" in da.dims:
            da = da.isel(ilev=0, drop=True)

        # 维度重命名
        rename_map = {}
        for d in da.dims:
            if d.lower().startswith("lon"):
                rename_map[d] = "lon"
            elif d.lower().startswith("lat"):
                rename_map[d] = "lat"
            elif d in ["lev_p", "level"]:
                rename_map[d] = "plev"
        if rename_map:
            da = da.rename(rename_map)

        # zonal mean
        if "lon" in da.dims:
            da = da.mean(dim="lon")

        # 60N
        if "lat" in da.dims:
            da = da.sel(lat=60.0, method="nearest")

        # 垂直维
        vd = None
        for cand in ["plev", "lev"]:
            if cand in da.dims:
                vd = cand
                break
        if vd is None:
            print(f"    [ERROR] No vertical dim in {fpath.name}, dims={da.dims}")
            ds.close()
            return None

        if vd != "lev":
            da = da.rename({vd: "lev"})

        # 单位统一为 Pa
        lev_vals = da.lev.values
        if np.nanmax(lev_vals) <= 2000.0:
            da = da.assign_coords(lev=lev_vals * 100.0)
            da.lev.attrs["units"] = "Pa"

        da = da.squeeze()
        da = da.reset_coords(drop=True)

        ds.close()
        return da

    except Exception as e:
        print(f"    [ERROR] Reading {fpath.name}: {e}")
        return None


def load_case_ensemble_U(case_name: str) -> xr.DataArray | None:
    """
    读取某个 case 的所有成员 U_60N(time, lev)，返回：
      DataArray(member, time, lev)
    """
    case_dir = Path(HINDCAST_BASE_DIR) / case_name
    u_dir = case_dir / "U"
    if not u_dir.is_dir():
        print(f"[BlockC]  [U] WARNING: directory not found: {u_dir}")
        return None

    files = sorted(u_dir.glob("*.nc"))
    if not files:
        print(f"[BlockC]  [U] WARNING: no files in {u_dir}")
        return None

    print(f"[BlockC]  [U] Loading case {case_name} ({len(files)} members)")

    da_list = []
    for mid, f in enumerate(files, start=1):
        da = load_and_process_single_member(f, mid)
        if da is not None:
            da = da.expand_dims("member")
            da = da.assign_coords(member=[mid])
            da_list.append(da)

    if not da_list:
        print(f"[BlockC]  [U] No valid members for {case_name}")
        return None

    try:
        ens = xr.concat(da_list, dim="member")  # (member, time, lev)
        return ens
    except Exception as e:
        print(f"[BlockC]  [U] ERROR concat {case_name}: {e}")
        return None


# ---------------------------------------------------------------------
# 2. O3：EXTR 气候态 & sigma_O3
# ---------------------------------------------------------------------

def load_o3_climatology_and_std() -> Tuple[xr.DataArray, float]:
    """
    从 EXTR O3 部分柱文件中计算：
      - O3_clim(dayofyear) : 日气候态
      - sigma_O3 (float)   : EXTR 全部时间 O3' 的标准差
    """
    print(f"[BlockC] Reading O3 EXTR from: {O3_EXTR_PC_NC}")
    ds = xr.open_dataset(O3_EXTR_PC_NC)

    if "O3_pc_60_90N_30_70hPa" in ds.data_vars:
        da = ds["O3_pc_60_90N_30_70hPa"]
    else:
        # 兜底：取第一个变量
        vname = list(ds.data_vars)[0]
        print(f"[BlockC]  [O3] WARNING: 'O3_pc_60_90N_30_70hPa' not found, use '{vname}' instead.")
        da = ds[vname]

    da = da.squeeze()  # (time,)
    ds.close()

    # 日气候态
    o3_clim = da.groupby("time.dayofyear").mean("time")

    # 异常与 sigma
    doy = da.time.dt.dayofyear
    o3_clim_on_time = o3_clim.sel(dayofyear=doy)
    o3_anom = da - o3_clim_on_time
    sigma_o3 = float(o3_anom.std().values)

    print(f"[BlockC] O3 climatology ready, sigma_O3 = {sigma_o3:.3f}")
    return o3_clim, sigma_o3


def load_o3_member_seasonal_anoms_for_case(
    case_name: str,
    o3_clim: xr.DataArray,
    sigma_o3: float,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    对某个 hindcast case，基于 O3_scores_{case}.nc 中的 O3_cap(member,time)，
    计算每个成员的季节平均 O3' 标准化：

      O3_MAM_std(member) : 3–5 月 (Mar–May)
      O3_AM_std (member) : 4–5 月 (Apr–May)

    返回：
      (O3_MAM_std, O3_AM_std)，均为 np.ndarray(shape=(n_member,))
      若出错，则返回两个长度 0 的数组。
    """
    nc_path = os.path.join(HINDCAST_O3_SCORES_DIR, f"O3_scores_{case_name}.nc")
    if not os.path.exists(nc_path):
        print(f"[BlockC]  [O3] WARNING: O3_scores not found for {case_name}: {nc_path}")
        return np.array([]), np.array([])

    ds = xr.open_dataset(nc_path)
    if "O3_cap" not in ds.data_vars:
        print(f"[BlockC]  [O3] WARNING: 'O3_cap' missing in {nc_path}, vars={list(ds.data_vars)}")
        ds.close()
        return np.array([]), np.array([])

    o3_mem = ds["O3_cap"]  # (member, time)
    ds.close()

    # 气候态在相同 time 上
    doy = o3_mem.time.dt.dayofyear
    o3_clim_on_time = o3_clim.sel(dayofyear=doy)  # (time,)
    # 广播到 member 维
    o3_clim_b = o3_clim_on_time.broadcast_like(o3_mem)
    o3_anom_mem = o3_mem - o3_clim_b  # (member, time)

    months = o3_mem.time.dt.month

    # 3–5 月
    mask_MAM = months.isin([3, 4, 5])
    o3_MAM = o3_anom_mem.where(mask_MAM, drop=True)  # (member, time_MAM)
    if o3_MAM.sizes.get("time", 0) > 0:
        o3_MAM_mean = o3_MAM.mean("time")  # (member,)
        O3_MAM_std = (o3_MAM_mean / sigma_o3).values
    else:
        O3_MAM_std = np.full(o3_mem.sizes["member"], np.nan)

    # 4–5 月 (Apr–May)
    mask_AM = months.isin([4, 5])
    o3_AM = o3_anom_mem.where(mask_AM, drop=True)  # (member, time_AM)
    if o3_AM.sizes.get("time", 0) > 0:
        o3_AM_mean = o3_AM.mean("time")  # (member,)
        O3_AM_std = (o3_AM_mean / sigma_o3).values
    else:
        O3_AM_std = np.full(o3_mem.sizes["member"], np.nan)

    print(
        f"[BlockC]  [O3] case {case_name}: "
        f"n_member={o3_mem.sizes['member']}, "
        f"valid_MAM={np.isfinite(O3_MAM_std).sum()}, "
        f"valid_AM={np.isfinite(O3_AM_std).sum()}"
    )

    return O3_MAM_std, O3_AM_std


# ---------------------------------------------------------------------
# 3. U：EXTR 气候态 & sigma_U(lev)，再算 hindcast 成员的 U'(t,z)/sigma
# ---------------------------------------------------------------------

def load_u_climatology_and_std() -> Tuple[xr.DataArray, xr.DataArray]:
    """
    从 U_EXTR_60N_lev_time.nc 中：
      - U_clim(dayofyear, lev)
      - sigma_U(lev)   — EXTR 中所有时间 U' 的标准差
    """
    print(f"[BlockC] Reading U EXTR from: {U_EXTR_60N_NC}")
    ds = xr.open_dataset(U_EXTR_60N_NC)
    u_extr = ds["U_60N"]  # (time, lev)
    ds.close()

    u_clim = u_extr.groupby("time.dayofyear").mean("time")  # (doy, lev)

    doy = u_extr.time.dt.dayofyear
    u_clim_on_time = u_clim.sel(dayofyear=doy)
    u_anom = u_extr - u_clim_on_time  # (time, lev)
    sigma_u = u_anom.std("time")      # (lev,)

    print(f"[BlockC] U climatology ready, lev={u_clim.lev.size}")
    return u_clim, sigma_u


def load_u_member_anom_std_for_case(
    case_name: str,
    u_clim: xr.DataArray,
    sigma_u: xr.DataArray,
) -> xr.DataArray | None:
    """
    对某个 case，读取 U_60N(member,time,lev)，计算：
      U'_std(member,time,lev) = [U - U_clim(doy)] / sigma_U(lev)

    返回：
      DataArray(member,time,lev)
    """
    ens = load_case_ensemble_U(case_name)
    if ens is None:
        return None

    # 气候态在对应 time 上
    doy = ens.time.dt.dayofyear
    u_clim_on_time = u_clim.sel(dayofyear=doy)  # (time, lev)
    u_clim_b = u_clim_on_time.broadcast_like(ens)  # (member,time,lev)

    u_anom_mem = ens - u_clim_b  # (member,time,lev)

    # sigma_u(lev) 扩展到 (member,time,lev)
    sigma_u_b = sigma_u.broadcast_like(u_anom_mem)
    u_std = u_anom_mem / sigma_u_b

    return u_std  # (member,time,lev)


# ---------------------------------------------------------------------
# 4. 相关性计算：对每个 case（按 member），再合并 group
# ---------------------------------------------------------------------

def corr_1d(x: np.ndarray, y: np.ndarray) -> float:
    """
    简单相关系数，允许 NaN，返回 NaN/数值。
    """
    mask = np.isfinite(x) & np.isfinite(y)
    if mask.sum() < 3:
        return np.nan
    xx = x[mask]
    yy = y[mask]
    if np.std(xx) == 0 or np.std(yy) == 0:
        return np.nan
    return float(np.corrcoef(xx, yy)[0, 1])


def compute_corr_for_group(group_cfg: Dict,
                           u_clim: xr.DataArray,
                           sigma_u: xr.DataArray,
                           o3_clim: xr.DataArray,
                           sigma_o3: float) -> xr.Dataset | None:
    """
    对一个 group（例如 group1），计算：

      对每个 case：
        corr_MAM_case(case,time,lev)
        corr_AM_case (case,time,lev)

      合并所有 case 的成员：
        corr_MAM_group(time,lev)
        corr_AM_group (time,lev)

    返回 Dataset 或 None（若有效 case 太少）。
    """
    gid = group_cfg["group_id"]
    gname = group_cfg["group_name"]
    case_names = group_cfg["cases"]

    print(f"\n[BlockC] ==== Group {gid}: {gname} ====")

    # 存每个 case 的 U'(member,time,lev) 与 O3'(member)
    u_std_cases: List[xr.DataArray] = []
    o3_MAM_cases: List[np.ndarray] = []
    o3_AM_cases: List[np.ndarray] = []
    valid_case_names: List[str] = []

    for cname in case_names:
        print(f"[BlockC]  >>> Case {cname}")

        u_std = load_u_member_anom_std_for_case(cname, u_clim, sigma_u)
        if u_std is None:
            print(f"[BlockC]   -> skip {cname} (U failed)")
            continue

        O3_MAM_std, O3_AM_std = load_o3_member_seasonal_anoms_for_case(
            cname, o3_clim, sigma_o3
        )
        if O3_MAM_std.size == 0 or O3_AM_std.size == 0:
            print(f"[BlockC]   -> skip {cname} (O3 failed)")
            continue

        # 检查成员数是否一致
        n_mem_u = u_std.sizes["member"]
        n_mem_o3 = O3_MAM_std.shape[0]
        if n_mem_u != n_mem_o3:
            print(
                f"[BlockC]   -> WARNING: member mismatch for {cname}, "
                f"U has {n_mem_u}, O3 has {n_mem_o3}; use min."
            )
            n_min = min(n_mem_u, n_mem_o3)
            u_std = u_std.isel(member=slice(0, n_min))
            O3_MAM_std = O3_MAM_std[:n_min]
            O3_AM_std = O3_AM_std[:n_min]

        u_std_cases.append(u_std)
        o3_MAM_cases.append(O3_MAM_std)
        o3_AM_cases.append(O3_AM_std)
        valid_case_names.append(cname)

    n_case = len(u_std_cases)
    if n_case == 0:
        print(f"[BlockC]  WARNING: no valid cases in group {gid}, skip.")
        return None

    print(f"[BlockC]  Group {gid} valid cases: {valid_case_names}")

    # 先对所有 case 的 (time,lev) 做对齐（按实际时间 inner）
    u_aligned = xr.align(*u_std_cases, join="inner")  # list of DataArray(member,time,lev)
    # 公共 time, lev
    time = u_aligned[0].time
    lev = u_aligned[0].lev
    n_time = time.size
    n_lev = lev.size

    # 为每个 case 分别建立相关性数组
    corr_MAM_case = np.full((n_case, n_time, n_lev), np.nan, dtype=float)
    corr_AM_case  = np.full((n_case, n_time, n_lev), np.nan, dtype=float)

    # 合并用的 sample 数量 = sum_i n_member_i
    Uvals_cases: List[np.ndarray] = []
    O3_MAM_vec_cases: List[np.ndarray] = []
    O3_AM_vec_cases: List[np.ndarray] = []
    n_members_each: List[int] = []

    for i_case, (u_std_da, O3_MAM_std, O3_AM_std, cname) in enumerate(
        zip(u_aligned, o3_MAM_cases, o3_AM_cases, valid_case_names)
    ):
        # u_std_da: (member, time, lev)
        Uvals = u_std_da.values   # numpy
        n_mem = Uvals.shape[0]
        print(
            f"[BlockC]  Case {cname}: aligned shape = {Uvals.shape}, "
            f"members = {n_mem}, time = {n_time}, lev = {n_lev}"
        )

        Uvals_cases.append(Uvals)
        O3_MAM_vec_cases.append(np.asarray(O3_MAM_std))
        O3_AM_vec_cases.append(np.asarray(O3_AM_std))
        n_members_each.append(n_mem)

        # ===== 对该 case，本身 case 内按成员做相关 =====
        for it in range(n_time):
            for kz in range(n_lev):
                x = Uvals[:, it, kz]           # member
                yM = O3_MAM_vec_cases[i_case]  # member
                yA = O3_AM_vec_cases[i_case]   # member

                corr_MAM_case[i_case, it, kz] = corr_1d(x, yM)
                corr_AM_case[i_case, it, kz]  = corr_1d(x, yA)

    # ===== 合并所有实验成员作为一个大样本集，做 group-level 相关 =====
    corr_MAM_group = np.full((n_time, n_lev), np.nan, dtype=float)
    corr_AM_group  = np.full((n_time, n_lev), np.nan, dtype=float)

    for it in range(n_time):
        for kz in range(n_lev):
            x_all = []
            yM_all = []
            yA_all = []
            for Uvals, O3M, O3A in zip(Uvals_cases, O3_MAM_vec_cases, O3_AM_vec_cases):
                x_all.append(Uvals[:, it, kz])  # (member,)
                yM_all.append(O3M)
                yA_all.append(O3A)
            x_all = np.concatenate(x_all)
            yM_all = np.concatenate(yM_all)
            yA_all = np.concatenate(yA_all)

            corr_MAM_group[it, kz] = corr_1d(x_all, yM_all)
            corr_AM_group[it, kz]  = corr_1d(x_all, yA_all)

    # 成员数信息（供 BlockD 使用）
    n_members_each_arr = np.asarray(n_members_each, dtype=int)
    n_samples_group = int(n_members_each_arr.sum())

    # ===== 打包成 Dataset =====
    ds_out = xr.Dataset(
        coords={
            "case": ("case", valid_case_names),
            "time": time,
            "lev": lev,
        },
        data_vars={
            "corr_MAM_case": (("case", "time", "lev"), corr_MAM_case),
            "corr_AM_case":  (("case", "time", "lev"), corr_AM_case),
            "corr_MAM_group": (("time", "lev"), corr_MAM_group),
            "corr_AM_group":  (("time", "lev"), corr_AM_group),
            "n_member_case": ("case", n_members_each_arr),
            "n_samples_group": ((), n_samples_group),
        },
        attrs={
            "group_id": gid,
            "group_name": gname,
            "case_names": ",".join(valid_case_names),
            "note": (
                "corr_MAM_case: per-experiment correlation across members between "
                "U60N'(member,time,lev)/sigma_U and O3'(Mar–May mean)/sigma_O3; "
                "corr_AM_case: same but O3'(Apr–May mean). "
                "corr_MAM_group / corr_AM_group: same correlations but with "
                "all members from all experiments in the group merged together. "
                "n_member_case: number of members in each experiment; "
                "n_samples_group: total number of members in the group."
            ),
        },
    )

    return ds_out


# ---------------------------------------------------------------------
# 5. 主函数
# ---------------------------------------------------------------------

def main_blockC():
    print("[BlockC] Start computing member-wise U–O3 correlations (per experiment & grouped) ...")

    # 气候态 & 标准差
    o3_clim, sigma_o3 = load_o3_climatology_and_std()
    u_clim, sigma_u   = load_u_climatology_and_std()

    for g in GROUPS:
        gid = g["group_id"]
        gname = g["group_name"]

        ds_corr = compute_corr_for_group(g, u_clim, sigma_u, o3_clim, sigma_o3)
        if ds_corr is None:
            continue

        out_path = os.path.join(CORR_ROOT, f"UO3_corr_group{gid}.nc")
        if os.path.exists(out_path):
            os.remove(out_path)
        ds_corr.to_netcdf(out_path)
        print(f"[BlockC] Saved group {gid} ({gname}) correlations to: {out_path}")

    print("\n[BlockC] All groups processed. Done.")


if __name__ == "__main__":
    main_blockC()


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
BlockD — Time–height correlation maps between U'(60°N, p) and seasonal O3' anomalies
=====================================================================================

- 读取 BlockC 产生的相关性数据：
    /home/weiji/test_code/plots/hindcast/UO3_corr/UO3_corr_group{gid}.nc

- 对每个 group（0–6）绘制：
    * MAM:  U'(t, p) vs ensemble-mean O3'(Mar–May, 30–70 hPa, 60–90°N)
    * AM :  U'(t, p) vs ensemble-mean O3'(Apr–May, 30–70 hPa, 60–90°N)

- 每个 group 出一幅多子图：
    - 对 group 内的每个实验 case（chem, nocouple, v2, v3 ...）各画一幅
    - 若该 group 内所有 case 的 init month 一致，则再加一幅
      “All members combined (N=...)” 的综合相关性图；
      若 init month 不一致（例如 group0: 0008-01/02/03），
      则 **不绘制 combined 图**。

- 每个子图：
    * 填色：相关系数 r(time, p)，限定在 1–100 hPa
    * 单根黑色等值线：r = −0.75
    * （可选）斜线阴影：p < P_THRESH 的区域（由 SHADE_SIG 控制）
    * Y 轴为 Pressure (hPa)，倒轴（高空在上）

- 整幅图：
    * 右侧单独 colorbar（不压到子图）
    * 图内右上角统一图例（Contour 和显著阴影）
"""

import os
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
from scipy.stats import t as student_t

# --------------------------------------------------
# 配置
# --------------------------------------------------
CORR_DIR = "/home/weiji/test_code/plots/hindcast/UO3_corr"
FIG_DIR  = "/home/weiji/test_code/plots/hindcast/UO3_corr_figs"
os.makedirs(FIG_DIR, exist_ok=True)

# 画所有 0–6 group
GROUPS_TO_PLOT = {
    0: "group0_small_different_init_month",
    1: "group1_Feb_chem_vs_nochem",
    2: "group2_Mar_chem_vs_nochem",
    3: "group3_largepert_different_init_month",
    4: "group4_Mar_different_pert",
    5: "group5_Qpert_different_init_month",
    6: "group6_Feb_different_pert",
}

# 显著性阴影开关 & 阈值（当前要求：暂时关闭，阈值设为 0.025）
SHADE_SIG = False       # True → 画显著区阴影；False → 不画
P_THRESH = 0.025        # p-value 阈值（双侧检验）


# ---------------------------------------------
# 工具函数：时间转换（cftime -> python datetime）
# ---------------------------------------------
def to_py_datetime(time_da: xr.DataArray):
    """将 xarray 的 time（可能是 cftime）转换成 numpy.datetime64 方便 matplotlib 画图。"""
    return np.array([np.datetime64(str(t)) for t in time_da.values])


# ---------------------------------------------
# 工具函数：由 r 和 N 计算双侧 p 值
# ---------------------------------------------
def corr_to_p_two_sided(r: np.ndarray, n_samples: int) -> np.ndarray:
    """
    给定相关系数 r 和样本数 n，计算双侧 p 值：
        t = r * sqrt((n-2) / (1-r^2))
        p = 2 * sf(|t|)  (Student-t 分布)
    """
    r_arr = np.asarray(r, dtype=float)
    # 避免 r=±1 时除零
    r_clip = np.clip(r_arr, -0.999999, 0.999999)
    df = max(n_samples - 2, 1)
    t_val = r_clip * np.sqrt(df / (1.0 - r_clip ** 2))
    p_val = 2.0 * student_t.sf(np.abs(t_val), df=df)
    return p_val


# ---------------------------------------------
# 工具函数：从 case 名称提取 init month
# ---------------------------------------------
def extract_init_month(case_label: str) -> str | None:
    """
    从 case 名称中提取 init month，返回两位字符串如 "01", "02", "03"。

    支持的模式：
      - "0008-01"
      - "0008-02_NOCOUPL"
      - "0008-03_v2"
      - "0008-03_v3"
    """
    base = case_label
    for s in ["_NOCOUPL", "_v2", "_v3"]:
        if case_label.endswith(s):
            base = case_label[: -len(s)]
            break

    if "-" not in base:
        return None

    parts = base.split("-")
    if len(parts) < 2:
        return None

    month_code = parts[1]
    if len(month_code) == 2 and month_code.isdigit():
        return month_code
    return None


def all_cases_same_init_month(case_labels) -> bool:
    """
    判断一个 group 内所有 case 的 init month 是否一致：
      - 若至少有一个 case 解析出 month_code
      - 且所有非 None 的 month_code 相同，则返回 True；
      - 否则 False。
    """
    months = []
    for c in case_labels:
        mc = extract_init_month(c)
        if mc is not None:
            months.append(mc)
    if not months:
        return False
    return len(set(months)) == 1


# ---------------------------------------------
# 根据 case 名称和样本数，构造更“人话”的标题
# ---------------------------------------------
def build_case_title(case_label: str, n_samples: int) -> str:
    """
    解析 case 标签，例如：
      - "0008-01"
      - "0008-02"
      - "0008-02_NOCOUPL"
      - "0008-03_v2"
      - "0008-03_v3"

    规则：
      * 年份固定从前四位取（0008）
      * init 月份从后两位取（01=Jan, 02=Feb, 03=Mar, 04=Apr）
      * 后缀：
          - 无后缀：small T pert, chem
          - _v2    : large T pert, chem
          - _v3    : Q pert, chem
          - _NOCOUPL: small T pert, prescribed O3 (NOCOUPL)
    """
    base = case_label
    suffix = None
    for s in ["_NOCOUPL", "_v2", "_v3"]:
        if case_label.endswith(s):
            base = case_label[: -len(s)]
            suffix = s
            break

    # base 形如 "0008-02"
    year = "0008"
    month_code = None
    if "-" in base:
        parts = base.split("-")
        if len(parts) >= 2:
            year = parts[0]
            month_code = parts[1]

    # 月份映射
    month_map = {
        "01": "Jan",
        "02": "Feb",
        "03": "Mar",
        "04": "Apr",
        "05": "May",
        "06": "Jun",
        "07": "Jul",
        "08": "Aug",
        "09": "Sep",
        "10": "Oct",
        "11": "Nov",
        "12": "Dec",
    }
    if month_code is None:
        month_str = "init"
    else:
        month_str = month_map.get(month_code, f"Mon {month_code}")

    init_str = f"{year} {month_str} init"

    # 扰动 / 耦合说明
    if suffix == "_NOCOUPL":
        pert_str = "small T pert, prescribed O$_3$ (NOCOUPL)"
    elif suffix == "_v2":
        pert_str = "large T pert, chem"
    elif suffix == "_v3":
        pert_str = "Q pert, chem"
    else:
        pert_str = "small T pert, chem"

    title = f"{init_str}: {pert_str} (N={n_samples})"
    return title


# ---------------------------------------------
# 单个子图的绘制函数
# ---------------------------------------------
def plot_single_panel(
    ax,
    time_da: xr.DataArray,
    lev_pa: np.ndarray,
    C: np.ndarray,
    title: str,
    n_samples: int | None = None,
    vmin: float = -1.0,
    vmax: float = 1.0,
    cmap_name: str = "RdBu_r",
    show_xlabel: bool = True,
    show_ylabel: bool = True,
):
    """
    在单个 axes 上画时间–高度相关性填色图：

    - 背景：r(time, p) 填色
    - 黑色等值线：r = -0.75
    - （可选）斜线阴影：p < P_THRESH 的区域（若 SHADE_SIG=True 且提供 n_samples）
    """

    # --- 时间转换 ---
    t_py = to_py_datetime(time_da)

    # --- lev: Pa -> hPa，并限制在 1–100 hPa ---
    lev_hpa = lev_pa / 100.0
    mask_lev = (lev_hpa >= 1.0) & (lev_hpa <= 100.0)
    lev_hpa_plot = lev_hpa[mask_lev]
    if mask_lev.sum() == 0:
        ax.text(0.5, 0.5, "No levels in 1–100 hPa", ha="center", va="center")
        ax.set_axis_off()
        return None

    # --- 对应裁剪 C 的垂直维 ---
    if C.shape[0] == time_da.size and C.shape[1] == lev_pa.size:
        # (time, lev)
        C_plot = C[:, mask_lev].T  # -> (lev, time)
    else:
        # (lev, time)
        C_plot = C[mask_lev, :]

    # --- 遮 NaN ---
    cmap = plt.get_cmap(cmap_name).copy()
    cmap.set_bad("lightgrey")
    C_masked = np.ma.masked_invalid(C_plot)

    # --- 主填色图 ---
    pcm = ax.pcolormesh(
        t_py,
        lev_hpa_plot,
        C_masked,
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
        shading="nearest",
    )

    # ------------------------------------------------------
    # 添加单根等值线：r = -0.75 （图例在整幅图上统一给出）
    # ------------------------------------------------------
    contour_level = [-0.75]
    ax.contour(
        t_py,
        lev_hpa_plot,
        C_masked,
        levels=contour_level,
        colors="black",
        linewidths=0.8,
    )

    # ------------------------------------------------------
    # 显著性阴影：p < P_THRESH（双侧检验）
    # ------------------------------------------------------
    if SHADE_SIG and (n_samples is not None) and (n_samples >= 4):
        p_val = corr_to_p_two_sided(C_plot, n_samples=n_samples)
        sig_mask = np.where(
            (p_val < P_THRESH) & np.isfinite(p_val), 1.0, 0.0
        )

        ax.contourf(
            t_py,
            lev_hpa_plot,
            sig_mask,
            levels=[0.5, 1.5],     # 只画值为 1 的区域
            colors="none",
            hatches=["////"],
        )

    # --- Y 轴、标题等 ---
    ax.invert_yaxis()
    ax.set_title(title, fontsize=11)

    if show_ylabel:
        ax.set_ylabel("Pressure (hPa)", fontsize=11)
    else:
        ax.set_yticklabels([])

    if show_xlabel:
        ax.set_xlabel("Calendar date (mapped to year 0008)", fontsize=11)
    else:
        ax.set_xticklabels([])

    # X 轴标月
    ax.xaxis.set_major_locator(mdates.MonthLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b"))
    for lab in ax.get_xticklabels():
        lab.set_rotation(30)
        lab.set_horizontalalignment("right")

    ax.grid(True, alpha=0.2)

    return pcm


# ---------------------------------------------
# 主绘图函数：对一个 group 画多子图
# ---------------------------------------------
def plot_group_time_height_corr(ds: xr.Dataset, gid: int, gname: str, season_tag: str):
    """
    对单个 group 画 time–height correlation 图集（多子图）：

    - ds: 包含 corr_MAM_case / corr_AM_case / corr_MAM_group / corr_AM_group
    - gid, gname: group id 和名字
    - season_tag: "MAM" 或 "AM"
    """

    # 选择变量名
    var_case = f"corr_{season_tag}_case"
    var_group = f"corr_{season_tag}_group"

    if var_case not in ds.data_vars:
        print(f"[BlockD] WARNING: {var_case} not in Dataset for group {gid}, skip.")
        return

    time = ds["time"]
    lev = ds["lev"].values  # Pa
    case_labels = [str(c) for c in ds["case"].values]

    # 决定是否画 combined panel：变量存在且 init month 一致才画
    same_init = all_cases_same_init_month(case_labels)
    has_group_var = var_group in ds.data_vars
    draw_group_panel = has_group_var and same_init

    # 需要绘多少面板：每个 case 一幅 + （可能存在）group 综合一幅
    n_case = len(case_labels)
    n_panels = n_case + (1 if draw_group_panel else 0)

    # 排版：最多 2 列
    ncols = 2
    nrows = int(np.ceil(n_panels / ncols))

    fig, axes = plt.subplots(
        nrows=nrows,
        ncols=ncols,
        figsize=(4.0 * ncols + 1.5, 3.0 * nrows + 1.0),
        squeeze=False,
        sharex=True,
        sharey=True,
    )

    pcms = []

    # ------- 辅助：样本数获取函数 -------

    def get_n_samples_for_case(case_label: str) -> int:
        """
        从 ds 中查找 case 的样本数：
        - 优先使用 n_samples_case(case)，若不存在再用 n_member_case(case)
        - 若都没有，则默认 30。
        """
        if "n_samples_case" in ds:
            try:
                return int(ds["n_samples_case"].sel(case=case_label).item())
            except Exception:
                pass
        if "n_member_case" in ds:
            try:
                return int(ds["n_member_case"].sel(case=case_label).item())
            except Exception:
                pass
        # fallback
        return 30

    def get_n_samples_for_group() -> int:
        """
        group 综合样本数：
        - 若 ds 中有 n_samples_group，用之；
        - 否则使用各 case 样本数之和。
        """
        if "n_samples_group" in ds:
            try:
                return int(ds["n_samples_group"].item())
            except Exception:
                pass
        return int(sum(get_n_samples_for_case(c) for c in case_labels))

    # ------- 逐 case 画图 -------

    panel_idx = 0
    for case_label in case_labels:
        row = panel_idx // ncols
        col = panel_idx % ncols
        ax = axes[row, col]

        C_case = ds[var_case].sel(case=case_label)  # (time, lev)
        n_samples_case = get_n_samples_for_case(case_label)

        # 包含 N 的标题
        title = build_case_title(case_label, n_samples_case)

        show_xlabel = (row == nrows - 1)
        show_ylabel = (col == 0)

        pcm = plot_single_panel(
            ax=ax,
            time_da=time,
            lev_pa=lev,
            C=C_case.values,
            title=title,
            n_samples=n_samples_case,
            vmin=-1.0,
            vmax=1.0,
            cmap_name="RdBu_r",
            show_xlabel=show_xlabel,
            show_ylabel=show_ylabel,
        )
        if pcm is not None:
            pcms.append(pcm)

        panel_idx += 1

    # ------- group 综合面板（若需要且允许绘制） -------
    if draw_group_panel:
        row = panel_idx // ncols
        col = panel_idx % ncols
        ax = axes[row, col]

        C_group = ds[var_group]  # (time, lev)
        n_samples_group = get_n_samples_for_group()

        title = f"All members combined (N={n_samples_group})"

        show_xlabel = (row == nrows - 1)
        show_ylabel = (col == 0)

        pcm = plot_single_panel(
            ax=ax,
            time_da=time,
            lev_pa=lev,
            C=C_group.values,
            title=title,
            n_samples=n_samples_group,
            vmin=-1.0,
            vmax=1.0,
            cmap_name="RdBu_r",
            show_xlabel=show_xlabel,
            show_ylabel=show_ylabel,
        )
        if pcm is not None:
            pcms.append(pcm)

        panel_idx += 1

    # 删除未用到的空子图
    while panel_idx < nrows * ncols:
        row = panel_idx // ncols
        col = panel_idx % ncols
        axes[row, col].set_axis_off()
        panel_idx += 1

    # ------- 整幅图标题（只保留物理含义） -------
    if season_tag == "MAM":
        season_desc = "Mar–May mean O3' (30–70 hPa, 60–90°N)"
    elif season_tag == "AM":
        season_desc = "Apr–May mean O3' (30–70 hPa, 60–90°N)"
    else:
        season_desc = f"{season_tag} mean O3' (30–70 hPa, 60–90°N)"

    fig.suptitle(
        f"Correlation between U'(60°N, p) and {season_desc}",
        fontsize=13,
    )

    # ------- 调整子图区域（给右侧色标留出空间，避免标题压到子图） -------
    fig.subplots_adjust(
        left=0.08,
        right=0.86,
        bottom=0.08,
        top=0.90,
        wspace=0.12,
        hspace=0.25,
    )

    # ------- 统一 colorbar：放到最右侧，右移，不与子图重叠 -------
    if pcms:
        cbar_ax = fig.add_axes([0.88, 0.18, 0.02, 0.6])
        cbar = fig.colorbar(
            pcms[0],
            cax=cbar_ax,
            orientation="vertical",
        )
        cbar.set_label("Correlation r", fontsize=11)

    # ------- 图例：Contour & （可选）显著阴影，放在主图内 -------
    legend_handles = []

    # r = -0.75 等值线
    contour_handle = Line2D(
        [0], [0],
        color="black",
        linewidth=1.0,
        label="Contour: r = −0.75",
    )
    legend_handles.append(contour_handle)

    # 显著性阴影图例（仅当 SHADE_SIG=True 时添加）
    if SHADE_SIG:
        hatch_patch = mpatches.Patch(
            facecolor="none",
            edgecolor="k",
            hatch="////",
            label=f"Significant at p < {P_THRESH:.3f} (two-sided)",
        )
        legend_handles.append(hatch_patch)

    if legend_handles:
        fig.legend(
            handles=legend_handles,
            loc="upper right",
            bbox_to_anchor=(0.86, 0.88),
            fontsize=9,
            frameon=True,
        )

    # ------- 保存 -------
    out_prefix = f"UO3_corr_{season_tag}_group{gid}_{gname}"
    png_path = os.path.join(FIG_DIR, out_prefix + ".png")
    pdf_path = os.path.join(FIG_DIR, out_prefix + ".pdf")
    fig.savefig(png_path, dpi=300)
    fig.savefig(pdf_path, dpi=300)
    plt.close(fig)

    print(f"[BlockD] Saved figure for group {gid}, season={season_tag} to:")
    print(f"         {png_path}")
    print(f"         {pdf_path}")


# ---------------------------------------------
# BlockD 主入口
# ---------------------------------------------
def main_blockD():
    print("[BlockD] Start plotting time–height correlation maps ...")
    print(f"[BlockD] SHADE_SIG = {SHADE_SIG}, P_THRESH = {P_THRESH}")

    for gid, gname in GROUPS_TO_PLOT.items():
        nc_path = os.path.join(CORR_DIR, f"UO3_corr_group{gid}.nc")
        if not os.path.exists(nc_path):
            print(f"[BlockD] WARNING: file not found for group {gid}: {nc_path}")
            continue

        print(f"\n[BlockD] === Group {gid}: reading {nc_path} ===")
        ds = xr.open_dataset(nc_path)
        print(f"[BlockD] group_name = {gname}")

        # 1) MAM: U'(t, p) vs O3'(Mar–May)
        plot_group_time_height_corr(
            ds=ds,
            gid=gid,
            gname=gname,
            season_tag="MAM",
        )

        # 2) AM: U'(t, p) vs O3'(Apr–May)
        plot_group_time_height_corr(
            ds=ds,
            gid=gid,
            gname=gname,
            season_tag="AM",
        )

        ds.close()

    print("\n[BlockD] All plots done.")


if __name__ == "__main__":
    main_blockD()
